# Post-Operative Patient Monitoring: Rule Engine Evaluation Notebook

**System Title:** Responsible AI-based Post-Operative Patient Monitoring & Escalation Workflow

**Scope:** Final 100% Evaluation Milestone (Prior Phase: 70% Milestone)

**Purpose:**
This notebook provides a complete, deterministic, and reproducible evaluation of the transparent, deterministic rule-based triage engine across six critical post-operative and telemetry scenarios:
1. **Normal** (Standard baseline recovery)
2. **High-risk** (Severe pain, fever, bleeding, or purulent drainage)
3. **Missing data** (Detached patch or telemetry drop)
4. **Noisy data** (Sensor artifacts or unphysiological telemetry)
5. **Conflicting data** (Baseline sensor vitals discordant with severe fever/chills symptoms)
6. **Communication/fallback failure** (Gateway offline engaging store-and-forward fallback)

> **Clinical & Safety Disclaimer:** This system is an academic research decision-support prototype. It provides deterministic clinical decision support only and does NOT make autonomous medical diagnoses. Final decisions remain strictly with licensed clinicians.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

# Enable inline plotting
%matplotlib inline

from data.generate_data import evaluate_dataset
print('Imports successful!')

## 1. Dataset Overview

Loading the synthesized post-operative observation dataset containing explicit ground-truth labels (`category`). The dataset is generated deterministically with fixed random seed (42).

In [ ]:
data_path = '../data/simulated_observations.csv'
df = pd.read_csv(data_path)
print(f'Total records in simulated dataset: {len(df)}')
print('\nDataset Schema:')
print(df.dtypes)
df.head()

In [ ]:
# Ground-Truth Category Distribution
print('Ground-Truth Category Distribution:')
scenario_dist = df['category'].value_counts()
print(scenario_dist)

# Missing/Noisy sensor and communication breakdown breakdown
print('\nData Quality Breakdown:')
print(f'Missing Temperature count: {df["temperature"].isna().sum()}')
print(f'Noisy Sensor count: {(df["sensor_status"] == "Noisy").sum()}')
print(f'Communication Failure count: {(df["communication_status"] == "Failed").sum()}')

plt.figure(figsize=(8, 4))
sns.countplot(data=df, y='category', palette='viridis')
plt.title('Distribution of Evaluation Scenarios')
plt.xlabel('Record Count')
plt.show()

## 2. Rule Engine Evaluation

Executing the deterministic triage policy on every observation record to produce `predicted_category` and `predicted_risk_level`.
We compare the true scenario (`category`) directly against `predicted_category`.

In [ ]:
evaluated_df = evaluate_dataset(df)
evaluated_df[['patient_code', 'category', 'predicted_category', 'predicted_risk_level', 'reason']].head(10)

In [ ]:
y_true = evaluated_df['category']
y_pred = evaluated_df['predicted_category']
labels = sorted(list(y_true.unique()))

report_str = classification_report(y_true, y_pred, labels=labels, digits=4)
print('=== CLASSIFICATION REPORT (Per-Class Performance) ===')
print(report_str)

report_dict = classification_report(y_true, y_pred, labels=labels, output_dict=True)
metrics_df = pd.DataFrame(report_dict).transpose()
metrics_df

## 3. Confusion Matrix Covering All 6 Scenarios

The confusion matrix compares ground-truth scenario on the vertical axis against the rule engine's predicted scenario on the horizontal axis.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels,
    square=True,
    linewidths=1,
    annot_kws={'size': 13, 'weight': 'bold'}
)
plt.title('Rule Engine Evaluation: 6x6 Confusion Matrix', fontsize=14, weight='bold')
plt.xlabel('Predicted Category', fontsize=12, weight='bold')
plt.ylabel('Ground-Truth Category', fontsize=12, weight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Threshold-Tuning Rationale & Boundary Analysis

| Vital / Signal | Cut-off Threshold | Clinical Risk Level | Rationale & Validation Context | Boundary Behavior & Limitations |
| :--- | :--- | :--- | :--- | :--- |
| **Body Temperature** | $\ge 38.5^\circ\text{C}$ or $< 35.0^\circ\text{C}$ | `HIGH` | Post-op pyrexia indicates possible acute surgical site infection or sepsis. Hypothermia indicates shock. Validated in simulation. | Borderline values ($38.4^\circ\text{C}$) trigger `MEDIUM` monitoring. Real-world validation requires clinical trial. |
| **Elevated Temp** | $37.5^\circ\text{C} - 38.4^\circ\text{C}$ | `MEDIUM` | Low-grade post-operative inflammatory response. Expected in 24-48h post-op. | Reassessed every 4-6 hours. Must be interpreted with symptom context. |
| **Pain Score** | $\ge 7 / 10$ | `HIGH` | Breakthrough or severe pain unresponsive to analgesics requires prompt surgical evaluation for hematoma or dehiscence. | Borderline 6/10 is `MEDIUM`. Subjective self-reporting has inter-patient variability. |
| **Moderate Pain** | $4 - 6 / 10$ | `MEDIUM` | Expected managed surgical pain requiring analgesic titration or monitoring. | May fluctuate with posture and ambulation. |
| **Wound Status** | `Discharge`, `Bleeding` | `HIGH` | Direct physical sign of wound infection (purulence) or active vascular compromise. | Must be physically confirmed by clinician. |
| **Wound Concern** | `Redness`, `Swelling` | `MEDIUM` | Localized inflammatory recovery phase; monitored for perimeter spreading. | Requires comparative serial wound inspection. |
| **Noisy Sensor** | Critical temp breach | `HIGH` | **Safety-First Guarantee:** Critical threshold breaches on noisy channels are NEVER ignored. | Triggers high severity with sensor quality limitation warning. |

> **Notice on Clinical Validation:** These thresholds reflect standard post-operative literature guidelines adapted for this academic simulation. They have been validated within this simulated testbed. Full clinical deployment would require multi-center prospective validation and IRB oversight.